In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

print("--- Starting Gold Layer: `fact_signal_readings` ---")

# 1. Read source data: OpenSignal conformed observations and current active dim_site
# Schema: device_id, site_ref, network_type, signal_dbm, measured_at, latitude, longitude, ...
df_signals = spark.read.table("inlap.silver.opensignal_conformed")

# dim_site for optional downstream validation / enrichment (keyed by master_site_id)
df_dim_site = spark.read.table("inlap.gold.dim_site").filter(F.col("is_current") == True)

# 2. Build unified site_ref → master_site_id lookup via sites_matched_registry.
#    RESOLVED sites (single_glid / ambiguous_multi_unit): master_site_id = resolved_glid
#    UNRESOLVED sites (no_match): master_site_id = UNRESOLVED_<sha2(lat_rounded|lon_rounded)>
#    This mirrors the synthetic key formula used in dim_site Stage 2e so readings for
#    unresolved physical locations link to their dim row instead of being treated as orphans.
df_registry_lookup = (
    spark.read.table("inlap.silver.sites_matched_registry")
    .filter(F.col("source_vendor") == "opensignal")
    .withColumn(
        "master_site_id",
        F.when(
            F.col("geolink_match_status") == "no_match",
            F.concat(
                F.lit("UNRESOLVED_"),
                F.sha2(
                    F.concat_ws(
                        "|",
                        F.col("lat_rounded").cast("string"),
                        F.col("lon_rounded").cast("string"),
                    ),
                    256,
                ),
            ),
        ).otherwise(F.col("resolved_glid")),
    )
    .select(
        F.col("site_id").alias("lookup_site_id"),
        F.col("master_site_id"),
    )
    .distinct()
)

# 3. Join signal readings to the unified lookup on site_ref = lookup_site_id
df_joined = df_signals.join(
    df_registry_lookup,
    df_signals["site_ref"] == df_registry_lookup["lookup_site_id"],
    "left",
)

# 4. Separate valid readings from true orphans.
#    True orphans = site_ref absent from sites_matched_registry for opensignal entirely.
#    Unresolved sites now resolve to a synthetic master_site_id and are NOT orphans.
df_valid_readings = df_joined.filter(F.col("master_site_id").isNotNull())
df_orphan_readings = df_joined.filter(F.col("master_site_id").isNull())

# 5. Append orphan readings to quarantine before excluding them from the fact build.
#    Surfaces referential-integrity failures instead of silently dropping them.
orphan_count = df_orphan_readings.count()
if orphan_count > 0:
    df_quarantine_rows = df_orphan_readings.select(
        F.to_json(
            F.struct(
                F.col("site_ref"),
                F.col("network_type"),
                F.col("signal_dbm"),
                F.col("measured_at"),
                F.col("latitude"),
                F.col("longitude"),
            )
        ).alias("raw_record"),
        F.lit("orphan - no matching dim_site at load time").alias("failure_reason"),
        F.lit("opensignal").alias("source_vendor"),
        F.current_timestamp().alias("quarantine_timestamp"),
    )
    df_quarantine_rows.write.format("delta").mode("append").saveAsTable("inlap.silver.quarantine_records")
    print(f"  Quarantined {orphan_count} orphan reading(s) → inlap.silver.quarantine_records")
else:
    print("  No orphan readings — all site_refs resolved or mapped to unresolved-synthetic keys.")

# 6. Aggregate daily signal metrics per site.
#    Use opensignal_conformed.network_type (the signal observation's own network type).
df_fact_incoming = (
    df_valid_readings
    .groupBy(
        F.col("master_site_id"),
        F.to_date(F.col("measured_at")).alias("reading_date"),
        F.col("network_type"),
    )
    .agg(
        F.avg("signal_dbm").alias("avg_signal_dbm"),
        F.min("signal_dbm").alias("min_signal_dbm"),
        F.max("signal_dbm").alias("max_signal_dbm"),
        F.count("*").alias("reading_count"),
    )
    .withColumn("fact_row_id", F.expr("uuid()"))
    .withColumn("_last_refreshed", F.current_timestamp())
)

# 7. MERGE into Gold Fact Table (idempotent; safe to re-run without clobbering existing rows)
fact_table_name = "inlap.gold.fact_signal_readings"
spark.sql("CREATE DATABASE IF NOT EXISTS inlap.gold")

deltaTable = DeltaTable.forName(spark, fact_table_name)
(
    deltaTable.alias("tgt")
    .merge(
        df_fact_incoming.alias("src"),
        "tgt.master_site_id = src.master_site_id"
        " AND tgt.reading_date = src.reading_date"
        " AND tgt.network_type = src.network_type",
    )
    .whenMatchedUpdate(set={
        "avg_signal_dbm": "src.avg_signal_dbm",
        "min_signal_dbm": "src.min_signal_dbm",
        "max_signal_dbm": "src.max_signal_dbm",
        "reading_count": "src.reading_count",
        "_last_refreshed": "src._last_refreshed",
    })
    .whenNotMatchedInsertAll()
    .execute()
)

fact_count = df_fact_incoming.count()
print(f"Successfully merged into `{fact_table_name}` with {fact_count} aggregated records.")
print(f"Orphan readings quarantined: {orphan_count}")

# Preview sample of the final gold fact table
display(spark.read.table(fact_table_name).orderBy(F.col("reading_date").desc()).limit(5))